# 09 · Layer 3：Coordination — 讓模型決定順序

上一章的流程是你寫死的。這一章把決定權交出去：

```
        使用者
           │
           ▼
    ┌─────────────┐
    │  總機 Agent  │  ← 看每個專員的 description，決定交給誰
    └──┬───┬───┬──┘
       ▼   ▼   ▼
     退款 物流 技術     ← 專員之間也可以互相轉手
```

ADK 的做法叫 **auto-delegation**：只要你給 agent 設了 `sub_agents`，
ADK 就會自動塞一個 `transfer_to_agent` 工具給它。模型「決定交給誰」的動作，
實際上就是呼叫這個工具。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. 建一支客服團隊

In [2]:
from google.adk.agents import LlmAgent


def check_refund(order_id: str) -> dict:
    """查詢退款進度。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "refund_status": "審核中", "expected_days": 3}


def track_package(order_id: str) -> dict:
    """查詢包裹目前位置。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "location": "桃園轉運中心", "eta": "明天下午"}


refund_agent = LlmAgent(
    name="refund_agent",
    model=get_model(),
    description="處理退貨與退款相關問題：查退款進度、說明退款政策、計算可退金額。",
    instruction="你是退款專員。用 check_refund 查詢，再用繁體中文簡短回覆。",
    tools=[check_refund],
)

shipping_agent = LlmAgent(
    name="shipping_agent",
    model=get_model(),
    description="處理物流配送問題：查包裹位置、預計送達時間、修改收件地址。",
    instruction="你是物流專員。用 track_package 查詢，再用繁體中文簡短回覆。",
    tools=[track_package],
)

tech_agent = LlmAgent(
    name="tech_agent",
    model=get_model(),
    description="處理產品技術問題：安裝設定、故障排除、韌體更新、相容性。",
    instruction="你是技術支援。用繁體中文給出具體的排除步驟，三步以內。",
)

front_desk = LlmAgent(
    name="front_desk",
    model=get_model(),
    instruction=(
        "你是客服總機。判斷使用者的問題屬於哪個專員的守備範圍，把工作轉交給他。"
        "不要自己回答專業問題。"
    ),
    sub_agents=[refund_agent, shipping_agent, tech_agent],
)

## 2. `transfer_to_agent` 是自動加上去的

我們沒有在 `front_desk` 的 `tools` 裡放任何東西：

In [3]:
print("front_desk.tools      :", front_desk.tools)
print("front_desk.sub_agents :", [a.name for a in front_desk.sub_agents])

front_desk.tools      : []
front_desk.sub_agents : ['refund_agent', 'shipping_agent', 'tech_agent']


但它在執行時會多一個 `transfer_to_agent` 工具。這個工具**不會出現在
`agent.tools` 裡**——它是由「流程（flow）」在組 request 時才注入的。

想確認一個 agent 有沒有交棒能力，看它用的是哪一種 flow：

| flow | 意義 |
|---|---|
| `AutoFlow` | 會注入 `transfer_to_agent`，**可以交棒** |
| `SingleFlow` | 不注入，**不能交棒** |

In [4]:
from google.adk.agents import LlmAgent as _LlmAgent

locked = _LlmAgent(
    name="locked", model=get_model(), instruction="i",
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,   # 兩個都關 + 沒有 sub_agents → SingleFlow
)

for a in (front_desk, refund_agent, locked):
    print(f"{a.name:16s} flow = {type(a._llm_flow).__name__}")

front_desk       flow = AutoFlow
refund_agent     flow = AutoFlow
locked           flow = SingleFlow


## 3. 跑三個不同類型的問題

注意 `trace=True` 印出來的 `transfer_to_agent` 呼叫——那就是「決策」發生的瞬間。

In [5]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=front_desk, app_name="concept_track")

questions = [
    "訂單 A-1001 的退款什麼時候會下來？",
    "我的包裹 B-2002 到哪了？",
    "產品開機之後燈一直閃紅色，怎麼辦？",
]

for q in questions:
    sid = await new_session(runner)
    print("=" * 60)
    print(f"Q: {q}")
    answer = await ask(runner, q, session_id=sid, trace=True)
    print(f"A: {answer}")

Q: 訂單 A-1001 的退款什麼時候會下來？


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'refund_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [refund_agent] 呼叫 check_refund({'order_id': 'A-1001'})
  ↩️  [refund_agent] check_refund 回傳 {'order_id': 'A-1001', 'refund_status': '審核中', 'expected_days': 3}


  💬 [refund_agent] 您的訂單 A-1001 目前退款審核中，預計將於 3 個工作天內完成。
A: 您的訂單 A-1001 目前退款審核中，預計將於 3 個工作天內完成。
Q: 我的包裹 B-2002 到哪了？


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'shipping_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [shipping_agent] 呼叫 track_package({'order_id': 'B-2002'})
  ↩️  [shipping_agent] track_package 回傳 {'order_id': 'B-2002', 'location': '桃園轉運中心', 'eta': '明天下午'}


  💬 [shipping_agent] 您的包裹 B-2002 目前位於桃園轉運中心，預計明天下午送達。
A: 您的包裹 B-2002 目前位於桃園轉運中心，預計明天下午送達。
Q: 產品開機之後燈一直閃紅色，怎麼辦？


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'tech_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  💬 [tech_agent] 產品出現紅燈閃爍通常代表發生錯誤，請依照以下步驟排除：

1. **斷電重啟**：拔掉電源線並等待 60 秒後重新插回，觀察紅燈是否消失。
2. **檢查連線**：確保所有連接埠（如電源線、網路線或傳輸線）皆已穩固插入。
3. **韌體/硬體重置**：若問題依舊，請查閱說明書執行「原廠設定重置 (Factory Reset)」或檢查官方網站是否有最新的韌體更新檔。
A: 產品出現紅燈閃爍通常代表發生錯誤，請依照以下步驟排除：

1. **斷電重啟**：拔掉電源線並等待 60 秒後重新插回，觀察紅燈是否消失。
2. **檢查連線**：確保所有連接埠（如電源線、網路線或傳輸線）皆已穩固插入。
3. **韌體/硬體重置**：若問題依舊，請查閱說明書執行「原廠設定重置 (Factory Reset)」或檢查官方網站是否有最新的韌體更新檔。


## 4. `description` 就是路由的依據

第 01 章說過 `description` 是寫給父 agent 看的。現在來證明它。

下面這組專員，**instruction 和工具完全不變**，只把 `description` 換成含糊的：

In [6]:
vague_refund = LlmAgent(
    name="refund_agent",
    model=get_model(),
    description="協助使用者。",
    instruction="你是退款專員。用 check_refund 查詢，再用繁體中文簡短回覆。",
    tools=[check_refund],
)

vague_shipping = LlmAgent(
    name="shipping_agent",
    model=get_model(),
    description="幫忙處理事情。",
    instruction="你是物流專員。用 track_package 查詢，再用繁體中文簡短回覆。",
    tools=[track_package],
)

vague_tech = LlmAgent(
    name="tech_agent",
    model=get_model(),
    description="回答問題。",
    instruction="你是技術支援。用繁體中文給出具體的排除步驟，三步以內。",
)

vague_desk = LlmAgent(
    name="front_desk",
    model=get_model(),
    instruction=(
        "你是客服總機。判斷使用者的問題屬於哪個專員的守備範圍，把工作轉交給他。"
        "不要自己回答專業問題。"
    ),
    sub_agents=[vague_refund, vague_shipping, vague_tech],
)

vague_runner = InMemoryRunner(agent=vague_desk, app_name="concept_track")

for q in questions:
    sid = await new_session(vague_runner)
    print("=" * 60)
    print(f"Q: {q}")
    print(f"A: {await ask(vague_runner, q, session_id=sid, trace=True)}")

Q: 訂單 A-1001 的退款什麼時候會下來？


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'refund_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [refund_agent] 呼叫 check_refund({'order_id': 'A-1001'})
  ↩️  [refund_agent] check_refund 回傳 {'order_id': 'A-1001', 'refund_status': '審核中', 'expected_days': 3}


  💬 [refund_agent] 訂單 A-1001 目前退款狀態為「審核中」，預計還需要 3 個工作天處理。
A: 訂單 A-1001 目前退款狀態為「審核中」，預計還需要 3 個工作天處理。
Q: 我的包裹 B-2002 到哪了？


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'shipping_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [shipping_agent] 呼叫 track_package({'order_id': 'B-2002'})
  ↩️  [shipping_agent] track_package 回傳 {'order_id': 'B-2002', 'location': '桃園轉運中心', 'eta': '明天下午'}


  💬 [shipping_agent] 您的包裹 B-2002 目前位於桃園轉運中心，預計明天下午送達。
A: 您的包裹 B-2002 目前位於桃園轉運中心，預計明天下午送達。
Q: 產品開機之後燈一直閃紅色，怎麼辦？


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'tech_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  💬 [tech_agent] 請嘗試以下步驟排除故障：

1. **強制重啟**：長按電源鍵 10-15 秒直到裝置完全關機，稍待片刻後再重新開機。
2. **檢查電源**：確保使用原廠充電線與轉接頭，並連接至穩定電源，確認是否有足夠電量。
3. **排除周邊**：移除所有連接的配件（如記憶卡或 USB 裝置），確認是否因硬體衝突導致異常。
A: 請嘗試以下步驟排除故障：

1. **強制重啟**：長按電源鍵 10-15 秒直到裝置完全關機，稍待片刻後再重新開機。
2. **檢查電源**：確保使用原廠充電線與轉接頭，並連接至穩定電源，確認是否有足夠電量。
3. **排除周邊**：移除所有連接的配件（如記憶卡或 USB 裝置），確認是否因硬體衝突導致異常。


含糊版的路由通常會亂掉——同一個問題可能被丟給錯的專員，或是總機自己回答了。

**結論：`description` 是多 agent 系統的路由表。** 它值得像寫 API 文件一樣認真寫。

## 5. 交棒之後，控制權還回得來嗎？

`LlmAgent` 有兩個開關管這件事：

| 參數 | 意思 |
|---|---|
| `disallow_transfer_to_parent` | 禁止轉回父 agent |
| `disallow_transfer_to_peers` | 禁止轉給兄弟 agent |

**⚠️ 網路上（包括不少中文教學）會告訴你這兩個預設是 `True`，
所以「交出去就回不來」。在 ADK 2.x 這是錯的。** 實際跑一次：

In [7]:
import google.adk

print(f"ADK 版本: {google.adk.__version__}\n")
print("實際預設值:")
for a in (refund_agent, shipping_agent, tech_agent):
    print(f"  {a.name:16s} disallow_to_parent={a.disallow_transfer_to_parent} "
          f"disallow_to_peers={a.disallow_transfer_to_peers}")

ADK 版本: 2.8.0

實際預設值:
  refund_agent     disallow_to_parent=False disallow_to_peers=False
  shipping_agent   disallow_to_parent=False disallow_to_peers=False
  tech_agent       disallow_to_parent=False disallow_to_peers=False


兩個都是 **`False`**——也就是**預設就允許雙向轉手**。

ADK 原始碼對 `disallow_transfer_to_parent` 的註解很值得完整讀一次：

> *Setting this as True also prevents this agent from continuing to reply to
> the end-user, and will transfer control back to the parent agent in the next
> turn. This behavior prevents one-way transfer, in which end-user may be stuck
> with one agent that cannot transfer to other agents in the agent tree.*

注意它說的是：設成 `True` 會讓控制權**在下一輪自動交還給父 agent**，
而這正是為了**避免**「使用者卡在某個 agent 手上」。

所以這兩個旗標不是「能不能交棒」的開關，而是「**由誰負責後續的路由**」。

### 驗證：交棒後再問一個別的領域的問題

In [8]:
sid = await new_session(runner)
print("第一個問題（物流）→ 應該轉給 shipping_agent:")
await ask(runner, "我的包裹 B-2002 到哪了？", session_id=sid, trace=True)

print("\n第二個問題（退款）→ 看 shipping_agent 會不會再轉一次:")
print(await ask(runner, "那我的退款呢？訂單 A-1001", session_id=sid, trace=True))

第一個問題（物流）→ 應該轉給 shipping_agent:


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'shipping_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [shipping_agent] 呼叫 track_package({'order_id': 'B-2002'})
  ↩️  [shipping_agent] track_package 回傳 {'order_id': 'B-2002', 'location': '桃園轉運中心', 'eta': '明天下午'}


  💬 [shipping_agent] 您的包裹 B-2002 目前位於桃園轉運中心，預計明天下午送達。

第二個問題（退款）→ 看 shipping_agent 會不會再轉一次:


  🔧 [shipping_agent] 呼叫 transfer_to_agent({'agent_name': 'refund_agent'})
  ↩️  [shipping_agent] transfer_to_agent 回傳 {'result': None}


  🔧 [refund_agent] 呼叫 check_refund({'order_id': 'A-1001'})
  ↩️  [refund_agent] check_refund 回傳 {'order_id': 'A-1001', 'refund_status': '審核中', 'expected_days': 3}


  💬 [refund_agent] 您的訂單 A-1001 目前退款進度為「審核中」，預計約需 3 個工作天處理。
您的訂單 A-1001 目前退款進度為「審核中」，預計約需 3 個工作天處理。


看第二題的 trace：發出 `transfer_to_agent` 的是 **`shipping_agent` 自己**。
專員直接把工作橫向轉給同事，總機沒有再介入。這就是預設值的效果。

### 反過來：把它鎖死會怎樣

現在把兩個旗標設成 `True`。先猜猜看第二題會發生什麼——
專員不能橫向轉手了，使用者會不會就卡住？

In [9]:
locked_refund = LlmAgent(
    name="refund_agent",
    model=get_model(),
    description="處理退貨與退款相關問題：查退款進度、說明退款政策、計算可退金額。",
    instruction="你是退款專員。用 check_refund 查詢，再用繁體中文簡短回覆。",
    tools=[check_refund],
    disallow_transfer_to_parent=True,   # ← 鎖死
    disallow_transfer_to_peers=True,
)

locked_shipping = LlmAgent(
    name="shipping_agent",
    model=get_model(),
    description="處理物流配送問題：查包裹位置、預計送達時間、修改收件地址。",
    instruction="你是物流專員。用 track_package 查詢，再用繁體中文簡短回覆。",
    tools=[track_package],
    disallow_transfer_to_parent=True,   # ← 鎖死
    disallow_transfer_to_peers=True,
)

locked_desk = LlmAgent(
    name="front_desk",
    model=get_model(),
    instruction="你是客服總機。把問題轉交給正確的專員，不要自己回答專業問題。",
    sub_agents=[locked_refund, locked_shipping],
)

print("鎖死之後的 flow:")
for a in (locked_refund, locked_shipping):
    print(f"  {a.name:16s} {type(a._llm_flow).__name__}")

locked_runner = InMemoryRunner(agent=locked_desk, app_name="concept_track")
sid = await new_session(locked_runner)

print("\n第一個問題（物流）:")
await ask(locked_runner, "我的包裹 B-2002 到哪了？", session_id=sid, trace=True)
print("\n第二個問題（退款）——注意這次是誰發出 transfer:")
print(await ask(locked_runner, "那我的退款呢？訂單 A-1001", session_id=sid, trace=True))

鎖死之後的 flow:
  refund_agent     SingleFlow
  shipping_agent   SingleFlow

第一個問題（物流）:


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'shipping_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [shipping_agent] 呼叫 track_package({'order_id': 'B-2002'})
  ↩️  [shipping_agent] track_package 回傳 {'order_id': 'B-2002', 'location': '桃園轉運中心', 'eta': '明天下午'}


  💬 [shipping_agent] 您的包裹 B-2002 目前位於「桃園轉運中心」，預計將於明天下午送達。

第二個問題（退款）——注意這次是誰發出 transfer:


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'refund_agent'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  🔧 [refund_agent] 呼叫 check_refund({'order_id': 'A-1001'})
  ↩️  [refund_agent] check_refund 回傳 {'order_id': 'A-1001', 'refund_status': '審核中', 'expected_days': 3}


  💬 [refund_agent] 您的訂單 A-1001 目前退款狀態為「審核中」，預計約需 3 個工作天處理，請您再耐心等待。
您的訂單 A-1001 目前退款狀態為「審核中」，預計約需 3 個工作天處理，請您再耐心等待。


### 結果：使用者**沒有**卡住

退款問題一樣被處理了，但**發出 `transfer_to_agent` 的人換了**：

| | 誰發出第二次 transfer | 路由形狀 |
|---|---|---|
| 預設（`False` / `False`） | **`shipping_agent` 自己** | 網狀：專員之間橫向轉手 |
| 鎖死（`True` / `True`） | **`front_desk`**（控制權已交還） | 星狀：每次都回總機重新分流 |

這正是原始碼註解說的：設成 `True` 之後，agent 不再繼續接手使用者，
下一輪控制權自動回到父 agent。

### 所以該怎麼選

| 你想要 | 設定 |
|---|---|
| 專員之間可以直接轉手，少一次往返 | 預設（`False`） |
| 所有路由決策都集中在總機，便於稽核與控制 | 兩個都設 `True` |

兩種都是合理設計，**沒有哪一種會讓使用者卡住**——前提是你的 agent 樹上
真的有一個「回得去」的父節點。真正會卡住的是：把一個**沒有父節點的
root agent** 的兩個旗標都設成 `True`（它會退化成 `SingleFlow`，
從此不能交棒也沒有人接手）。

## 6. 另一條路：`AgentTool`

`sub_agents` 是**交棒**——控制權真的轉移過去。
`AgentTool` 是**外包**——把另一個 agent 包成一個「工具」，
呼叫完結果回到原本的 agent 手上。

| | `sub_agents` | `AgentTool` |
|---|---|---|
| 控制權 | 轉移出去 | 留在原地 |
| 對話由誰接手 | 新的 agent | 還是原本的 agent |
| 適合 | 分流到不同專員 | 「我需要一個子答案來完成我的工作」 |

`AgentTool` 也是**繞過「內建工具不能混用」限制**的標準解法（第 02 章的伏筆）。

In [10]:
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

# 專門負責搜尋的 agent——它自己只有一個內建工具，合法
search_specialist = LlmAgent(
    name="search_specialist",
    model=get_model(),
    description="用 Google 搜尋取得最新公開資訊。",
    instruction="用搜尋找出答案，用繁體中文摘要重點，三句話以內。",
    tools=[google_search],
)

# 主 agent 有自己的函式工具 + 把搜尋 agent 當成一個工具
assistant = LlmAgent(
    name="assistant",
    model=get_model(),
    instruction=(
        "你是研究助理。需要查訂單就用 track_package；"
        "需要外部最新資訊就用 search_specialist 工具。用繁體中文回答。"
    ),
    tools=[track_package, AgentTool(agent=search_specialist)],
)

tool_runner = InMemoryRunner(agent=assistant, app_name="concept_track")
sid = await new_session(tool_runner)
try:
    print(await ask(tool_runner, "Google ADK 最新版本有什麼特色？", session_id=sid, trace=True))
except Exception as exc:
    print(f"⚠️ {type(exc).__name__}: {str(exc)[:160]}")
    print("\n（Search grounding 的免費配額跟一般模型呼叫是分開算的，而且緊很多。"
          "重點在於上面的組裝方式——AgentTool 讓內建工具與自訂工具能共存於同一個系統。）")

  🔧 [assistant] 呼叫 search_specialist({'request': 'Google ADK 最新版本特色'})


⚠️ _ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURC

（Search grounding 的免費配額跟一般模型呼叫是分開算的，而且緊很多。重點在於上面的組裝方式——AgentTool 讓內建工具與自訂工具能共存於同一個系統。）


注意事件串流裡的 author：`AgentTool` 的呼叫看起來就像一次普通的工具呼叫，
**控制權從來沒有離開 `assistant`**。這就是它跟 `sub_agents` 的根本差別。

## 7. Orchestration vs Coordination

| | Orchestration（第 08 章） | Coordination（本章） |
|---|---|---|
| 誰決定順序 | 你，寫在程式裡 | 模型 |
| 可預期性 | 每次都一樣 | 每次可能不同 |
| 應變能力 | 沒有 | 有 |
| 成本 | 低（不花 token 決策） | 高（每次交棒都是一次呼叫） |
| 除錯 | 容易 | 要看 trace |
| 適合 | 訂單處理、報表產生、ETL | 客服分流、開放式研究 |

**實務上兩者常常混用**：外層用 Orchestration 保證主流程可控，
某一站內部用 Coordination 處理開放性的子問題。

## 本章重點

- **設了 `sub_agents`，ADK 就自動給你 `transfer_to_agent` 工具**，
  模型的「決策」就是呼叫它。
- **`description` 是路由表**。含糊的 description 會讓路由整個亂掉。
- **⚠️ 交棒預設是雙向的**：`disallow_transfer_to_parent` /
  `disallow_transfer_to_peers` 在 ADK 2.x **預設都是 `False`**。
  很多資料說預設是 `True`「交出去回不來」，那是舊版或誤傳——本章實測過。
- **這兩個旗標決定的是「誰負責後續路由」，不是「能不能交棒」**：
  預設＝專員之間橫向轉手（網狀）；設成 `True`＝下一輪交還父 agent
  由總機重新分流（星狀）。兩種都不會讓使用者卡住。
- **`transfer_to_agent` 不在 `agent.tools` 裡**，是 flow 在執行時注入的；
  要判斷有沒有交棒能力，看 `AutoFlow` / `SingleFlow`。
- **`AgentTool` 是外包不是交棒**，控制權留在原地；
  它也是繞過「內建工具不能混用」的標準解法。
- **Orchestration 換可預期，Coordination 換應變能力**，實務上混用。

## 動手練習

1. 把 `tech_agent` 的 description 改成「處理所有問題」，
   重跑第 3 節，看它是不是把所有問題都吃走了。
2. 第 5 節只把 `disallow_transfer_to_peers` 設成 `True`（parent 保持 `False`），
   看跨領域的第二個問題會走「轉回總機再分流」還是直接卡住。
3. 把第 6 節的 `AgentTool` 改成 `sub_agents`，比較事件串流的差異。

---
**下一站 → `10_graph_workflows.ipynb`**：ADK 2.0 的圖形化執行引擎，
以及它怎麼補上「條件分支」這個缺口。